# M1 Notebook 30 — Mathematical Foundations Capstone

**Status:** Runnable first edition

## Capstone objective

Design an end-to-end national resource-allocation model combining data preparation, uncertainty, optimization, simulation, validation, and communication.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_math.case_studies import (
    normalized_scores,portfolio_risk,rank_options,scenario_loss,weighted_score,
)
from srai_math.optimization import project_simplex,projected_gradient_descent
from srai_math.randomized import random_project
from srai_math.information import entropy


## 1. Synthetic national indicator dataset

In [ ]:
rng=np.random.default_rng(2026)
regions=[f"Region_{i+1}" for i in range(12)]
indicators=pd.DataFrame({
    "Population_need":rng.uniform(50,100,12),
    "Service_gap":rng.uniform(20,90,12),
    "Implementation_capacity":rng.uniform(30,95,12),
    "Climate_risk":rng.uniform(10,100,12),
},index=regions)
indicators.head()


## 2. Standardization and composite need score

In [ ]:
Z=normalized_scores(indicators.to_numpy())
weights=np.array([.35,.30,.20,.15])
need_scores=weighted_score(Z,weights)
ranking=rank_options(need_scores,regions)
ranking[:5]


## 3. Uncertainty scenarios

In [ ]:
scenario_prob=np.array([.25,.5,.25])
scenario_multiplier=np.array([1.3,1.0,.8])
expected_multiplier=scenario_loss(scenario_prob,scenario_multiplier)
{"expected_multiplier":expected_multiplier,"uncertainty_entropy_bits":entropy(scenario_prob)}


## 4. Allocation optimization

In [ ]:
base_need=np.maximum(need_scores-need_scores.min()+.1,0)
returns=base_need/base_need.max()
cov=np.diag(rng.uniform(.4,1.2,12))
cov+=.05*np.ones((12,12))
objective=lambda w: float(.5*w@cov@w-1.2*returns@w)
gradient=lambda w: cov@w-1.2*returns
allocation,history,_=projected_gradient_descent(
    objective,gradient,np.full(12,1/12),
    lambda x: project_simplex(x,1.0),
    learning_rate=.15,max_iter=1000
)
allocation_series=pd.Series(allocation,index=regions).sort_values(ascending=False)
allocation_series.head()


## 5. Risk and sensitivity

In [ ]:
base_risk=portfolio_risk(allocation,cov)
alternative_weights=np.array([.25,.25,.25,.25])
alt_scores=weighted_score(Z,alternative_weights)
rank_change=pd.DataFrame({
    "base_rank":[r for r,_ in ranking],
    "equal_weight_rank":[r for r,_ in rank_options(alt_scores,regions)]
})
{"portfolio_risk":base_risk,"top_region":allocation_series.index[0]}


## 6. Dimensionality reduction for visualization

In [ ]:
compressed,R=random_project(Z,2,seed=8)
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(compressed[:,0],compressed[:,1],s=100*allocation+20)
for i,label in enumerate(regions):
    ax.text(compressed[i,0],compressed[i,1],label,fontsize=8)
ax.set_title("Regions in Random-Projection Space")
plt.show()


## 7. Validation checks

In [ ]:
checks={
    "allocation_sums_to_one":np.isclose(allocation.sum(),1.0),
    "allocation_nonnegative":bool(np.all(allocation>=0)),
    "objective_decreased":history[-1]<=history[0],
    "finite_risk":np.isfinite(base_risk),
}
checks


## 8. Executive decision table

In [ ]:
decision_table=pd.DataFrame({
    "Need_score":need_scores,
    "Recommended_allocation":allocation,
    "Expected_scenario_adjusted_allocation":allocation*expected_multiplier,
},index=regions).sort_values("Recommended_allocation",ascending=False)
decision_table.head(10)


## 9. Governance interpretation

The mathematical model informs allocation but does not replace policy judgment. Final decisions should include equity constraints, legal mandates, minimum service floors, political feasibility, implementation capacity, and stakeholder review.

## 10. Capstone deliverables

- Mathematical formulation
- Reproducible Python implementation
- Validation tests
- Sensitivity analysis
- Decision table
- Governance interpretation
- Professional reporting template

## Final M1 insight

Mathematical foundations become operational when algebra, calculus, probability, statistics, optimization, numerical methods, and information theory are combined into a validated decision system.